# Delta Lake Streaming

This notebook demonstrates how to use Delta Lake with streaming data. We'll:

1. Set up a streaming source that simulates real-time data
2. Configure a Delta Lake sink to write the streaming data
3. Demonstrate how Delta Lake handles schema evolution in streaming
4. Show how to query the Delta table while streaming is active
5. Explore streaming metrics and monitoring

## 1. Initialize Spark Session with Delta Lake

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_date, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType

# Set SPARK_HOME environment variable
os.environ['SPARK_HOME'] = '/usr/local/lib/python3.10/site-packages/pyspark'

# Add scripts directory to path
sys.path.append('/opt/spark/scripts')
try:
    import utils
except ImportError:
    print("Could not import utils module")

# Create Spark session with Delta Lake support
spark = utils.create_spark_session("Delta Lake Streaming")

print(f"Spark version: {spark.version}")
try:
    delta_version = spark.sql("SELECT version() as delta_version").collect()[0][0]
    print(f"Delta Lake version: {delta_version}")
except:
    print("Could not determine Delta Lake version")

## 2. Define Constants and Paths

In [ ]:
# Define paths
DATA_DIR = "/opt/spark/data"
DELTA_TABLE_PATH = os.path.join(DATA_DIR, "processed/global_superstore_delta")
STREAM_SOURCE_PATH = os.path.join(DATA_DIR, "streaming/source")
STREAM_CHECKPOINT_PATH = os.path.join(DATA_DIR, "streaming/checkpoint")

# Create directories if they don't exist
os.makedirs(STREAM_SOURCE_PATH, exist_ok=True)
os.makedirs(STREAM_CHECKPOINT_PATH, exist_ok=True)

print(f"Delta table path: {DELTA_TABLE_PATH}")
print(f"Stream source path: {STREAM_SOURCE_PATH}")
print(f"Stream checkpoint path: {STREAM_CHECKPOINT_PATH}")

## 3. Generate Streaming Data

In [ ]:
# Define schema for streaming data
stream_schema = StructType([
    StructField("Row ID", IntegerType(), False),
    StructField("Order ID", StringType(), False),
    StructField("Order Date", DateType(), False),
    StructField("Ship Date", DateType(), False),
    StructField("Ship Mode", StringType(), False),
    StructField("Customer ID", StringType(), False),
    StructField("Customer Name", StringType(), False),
    StructField("Segment", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("Sub-Category", StringType(), False),
    StructField("Sales", DoubleType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("Profit", DoubleType(), False),
    StructField("Source", StringType(), False),
    StructField("Timestamp", TimestampType(), False)
])

# Generate sample streaming data
def generate_stream_batch(batch_id, num_records=10):
    """Generate a batch of streaming data"""
    stream_data = utils.generate_test_data(num_records=num_records, scenario='normal')
    
    # Add streaming-specific columns
    stream_data['Source'] = f'stream_batch_{batch_id}'
    stream_data['Timestamp'] = pd.Timestamp.now()
    
    # Save to the streaming source directory
    output_path = os.path.join(STREAM_SOURCE_PATH, f"batch_{batch_id}.json")
    stream_data.to_json(output_path, orient='records', lines=True)
    
    print(f"Generated batch {batch_id} with {num_records} records at {output_path}")
    return output_path

# Generate initial batches
for i in range(3):
    generate_stream_batch(i, num_records=5)

## 4. Set Up Streaming Query

In [ ]:
# Read streaming data
stream_df = spark.readStream \
    .format("json") \
    .schema(stream_schema) \
    .option("maxFilesPerTrigger", 1) \
    .load(STREAM_SOURCE_PATH)

# Add processing timestamp
stream_df = stream_df.withColumn("Processing_Time", current_timestamp())

# Write to Delta table
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", STREAM_CHECKPOINT_PATH) \
    .trigger(processingTime="5 seconds") \
    .start(DELTA_TABLE_PATH)

print("Streaming query started. Will run for 30 seconds...")
import time
time.sleep(30)
query.stop()
print("Streaming query stopped.")

## 5. Generate More Streaming Data with Schema Evolution

In [ ]:
# Generate streaming data with schema evolution
def generate_evolved_batch(batch_id, num_records=10):
    """Generate a batch of streaming data with schema evolution"""
    stream_data = utils.generate_test_data(num_records=num_records, scenario='schema_change')
    
    # Add streaming-specific columns
    stream_data['Source'] = f'evolved_batch_{batch_id}'
    stream_data['Timestamp'] = pd.Timestamp.now()
    
    # Save to the streaming source directory
    output_path = os.path.join(STREAM_SOURCE_PATH, f"evolved_batch_{batch_id}.json")
    stream_data.to_json(output_path, orient='records', lines=True)
    
    print(f"Generated evolved batch {batch_id} with {num_records} records at {output_path}")
    return output_path

# Generate evolved batches
for i in range(3):
    generate_evolved_batch(i, num_records=5)

## 6. Stream with Schema Evolution

In [ ]:
# Read streaming data with schema evolution
evolved_stream_df = spark.readStream \
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .option("inferSchema", "true") \
    .load(STREAM_SOURCE_PATH)

# Add processing timestamp
evolved_stream_df = evolved_stream_df.withColumn("Processing_Time", current_timestamp())

# Write to Delta table with schema evolution
query = evolved_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", STREAM_CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .trigger(processingTime="5 seconds") \
    .start(DELTA_TABLE_PATH)

print("Streaming query with schema evolution started. Will run for 30 seconds...")
time.sleep(30)
query.stop()
print("Streaming query stopped.")

## 7. Query the Delta Table

In [ ]:
# Query the Delta table
delta_df = spark.read.format("delta").load(DELTA_TABLE_PATH)

# Show the schema
print("Delta Table Schema:")
delta_df.printSchema()

# Count records by source
print("\nRecord Count by Source:")
delta_df.groupBy("Source").count().orderBy("Source").show(truncate=False)

# Show sample data
print("\nSample Data:")
delta_df.orderBy(col("Timestamp").desc()).show(5, truncate=False)

## 8. Explore Delta Table History

In [ ]:
# Get Delta table history
print("Delta Table History:")
spark.sql(f"DESCRIBE HISTORY delta.`{DELTA_TABLE_PATH}`").show(truncate=False)

# Get Delta table details
print("\nDelta Table Details:")
spark.sql(f"DESCRIBE DETAIL delta.`{DELTA_TABLE_PATH}`").show(truncate=False)

## 9. Streaming Complete

In [ ]:
print("Delta Lake streaming demo completed successfully!")
print(f"Delta table is available at: {DELTA_TABLE_PATH}")
print("You can now proceed with the other notebooks in the demo.")